In [ ]:
# BLOQUE 1 — Configuración del catálogo y del lote
from pathlib import Path
from datetime import datetime, timezone
import json
import re
import sys
import uuid

# Cargo el repositorio completo; no uso rutas personales en el código.
dbutils.widgets.text("repo_root", "", "Raíz del repositorio en Workspace")
root_parameter = dbutils.widgets.get("repo_root").strip()
if root_parameter and not Path(root_parameter).is_absolute():
    raise ValueError("repo_root debe ser absoluto.")
current = Path.cwd().resolve()
roots = [Path(root_parameter)] if root_parameter else [current, *current.parents]
project_root = next((p for p in roots if (p / "databricks/gtfs_bronze/discovery_runtime.py").is_file()), None)
if project_root is None:
    raise RuntimeError("Sincroniza el repositorio completo y configura repo_root.")
module_path = str(project_root / "databricks")
if module_path not in sys.path:
    sys.path.insert(0, module_path)
from gtfs_bronze.contracts import quoted_table
from gtfs_bronze.discovery import fetch_catalog, normalize_catalog, validate_selection
from gtfs_bronze.discovery_runtime import create_ingestion_plan, emit_ingestion_plan

defaults = {"catalog": "workspace", "schema": "gtfs_bogota", "explicit_version": "",
            "start_date": "", "end_date": "", "max_versions": "1", "max_metadata_rows": "5000",
            "reprocess_completed": "false", "emit_task_values": "false"}
for key, default in defaults.items():
    dbutils.widgets.text(key, default)
dbutils.widgets.dropdown("selection_mode", "latest_uningested",
                         ["latest_uningested", "backfill_range", "explicit_version"])
parameters = {key: dbutils.widgets.get(key).strip() for key in defaults}
for key in ("reprocess_completed", "emit_task_values"):
    if parameters[key].lower() not in {"true", "false"}:
        raise ValueError(f"{key} debe ser true o false.")
catalog, schema = parameters["catalog"], parameters["schema"]
quoted_table(f"{catalog}.{schema}.configuration_check")
max_metadata_rows = int(parameters["max_metadata_rows"])
if not 1 <= max_metadata_rows <= 10000:
    raise ValueError("max_metadata_rows debe estar entre 1 y 10000.")
selection = {key: parameters[key] for key in ("explicit_version", "start_date", "end_date")}
selection.update(selection_mode=dbutils.widgets.get("selection_mode").strip(),
                 max_versions=int(parameters["max_versions"]),
                 reprocess_completed=parameters["reprocess_completed"].lower() == "true")
validate_selection(**selection)
source_config = json.loads((project_root / "databricks/config/static_gtfs_catalog.json").read_text(encoding="utf-8"))
gtfs_group_id = source_config["gtfs_group_id"]
if not re.fullmatch(r"[0-9a-f]{32}", gtfs_group_id):
    raise ValueError("El grupo ArcGIS configurado no es válido.")
catalog_endpoint = source_config["portal_base_url"].rstrip("/") + f"/sharing/rest/content/groups/{gtfs_group_id}/search"
control_table = f"{catalog}.{schema}.control_gtfs_static_catalog"
observation_table = f"{catalog}.{schema}.audit_gtfs_static_catalog_observations"
catalog_run_id = str(uuid.uuid4())
discovered_at_utc = datetime.now(timezone.utc)
spark.conf.set("spark.sql.session.timeZone", "UTC")
print(f"Corrida: {catalog_run_id}; modo: {selection['selection_mode']}")
print("Actualizo metadata del catálogo; no descargo ZIP ni cargo tablas Bronze/Silver.")


In [ ]:
# BLOQUE 2 — Consulta completa y paginada del catálogo público
import requests
catalog_items = fetch_catalog(
    catalog_endpoint, requests.get, max_items=max_metadata_rows, max_pages=source_config["max_pages"],
    retry_errors=(requests.ConnectionError, requests.Timeout),
)
print(f"Elementos descubiertos: {len(catalog_items)}")


In [ ]:
# BLOQUE 3 — Contrato del catálogo y esquema explícito
from pyspark.sql.types import BooleanType, StringType, StructField, StructType, TimestampType
catalog_rows = normalize_catalog(catalog_items, group_id=gtfs_group_id,
                                 run_id=catalog_run_id, discovered_at=discovered_at_utc)
# Conservo el esquema histórico de las tablas de control y observaciones.
catalog_schema = StructType([
    StructField("feed_version", StringType(), False),
    StructField("portal_item_id", StringType(), False),
    StructField("portal_title", StringType(), False),
    StructField("source_url", StringType(), True),
    StructField("is_direct_zip_url", BooleanType(), False),
    StructField("portal_item_type", StringType(), True),
    StructField("portal_created_at_utc", TimestampType(), True),
    StructField("portal_modified_at_utc", TimestampType(), True),
    StructField("metadata_fingerprint", StringType(), False),
    StructField("source_group_id", StringType(), False),
    StructField("catalog_run_id", StringType(), False),
    StructField("discovered_at_utc", TimestampType(), False),
])
catalog_df = spark.createDataFrame(catalog_rows, schema=catalog_schema)
display(catalog_df.orderBy("feed_version", ascending=False))


In [ ]:
# BLOQUE 4 — Estado del catálogo y observaciones append-only
from delta.tables import DeltaTable
if spark.catalog.tableExists(quoted_table(control_table)):
    (DeltaTable.forName(spark, quoted_table(control_table)).alias("target")
     .merge(catalog_df.alias("source"), "target.portal_item_id = source.portal_item_id")
     .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
else:
    catalog_df.writeTo(quoted_table(control_table)).using("delta").create()
# Una nueva corrida conserva otra observación; no modifica archivos Raw.
catalog_df.write.format("delta").mode("append").saveAsTable(quoted_table(observation_table))
print(f"Control: {control_table}; observaciones: {observation_table}")


In [ ]:
# BLOQUE 5 — Plan Bronze desde cierres completos y procedencia
plan = create_ingestion_plan(spark, catalog=catalog, schema=schema,
                             max_metadata_rows=max_metadata_rows, **selection)
print(f"Resultado: {plan['status']}; trabajos: {len(plan['work_items'])}; conteos: {plan['counts']}")
if plan["decisions"]:
    display(spark.createDataFrame(plan["decisions"]).orderBy("feed_version"))
else:
    print("El rango no contiene versiones; no hay trabajo de ingesta.")
print(json.dumps(plan["work_items"], ensure_ascii=False, indent=2))
print(f"Intentos excluidos mediante resoluciones vigentes: {len(plan['resolved_attempts'])}")


In [ ]:
# BLOQUE 6 — Entrega optativa al Workflow
# Run all vuelve a leer la selección y reconstruye el plan antes de entregarlo.
emit = dbutils.widgets.get("emit_task_values").strip().lower()
if emit not in {"true", "false"}:
    raise ValueError("emit_task_values debe ser true o false.")
if emit == "true":
    emit_ingestion_plan(dbutils, plan)
    print(f"Plan entregado al Job: {plan['status']}")
else:
    print("Vista previa del plan: no entregué task values. Los bloques 1–4 sí actualizaron metadata del catálogo.")
